In [10]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 32)
        self.fc2 = nn.Linear(32, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

class ReplayBuffer:
    def __init__(self, max_size):
        self.buffer = deque(maxlen=max_size)
    def add_batch(self,batch_size, tuplex):
        for i in range(batch_size):
            self.buffer.append(tuplex[i])
    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[idx] for idx in indices])
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)
    def __len__(self):
        return len(self.buffer)

In [26]:
class FrozenLakeEnv():
    ACTIONS = ['L', 'D', 'R', 'U']

    def __init__(self ,batch_size=32, gamma=0.99, epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995, step_count_rate=20):
        self.env = gym.make('FrozenLake-v1', is_slippery=False)
        self.state_dim = self.env.observation_space.n
        self.action_dim = self.env.action_space.n
        self.batch_size = batch_size
        self.gamma = gamma
        self.step_count_rate = step_count_rate
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

    def inputstate_tensor(self, state):
        state_tensor = torch.zeros(self.state_dim).to(device)
        state_tensor[state] = 1.0
        return state_tensor

    def optimize_model(self, target_net, replay_buffer, optimizer, loss_fn, batch_size, gamma):
        if len(replay_buffer) < batch_size:
            return

        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        states_tensor = torch.stack([self.inputstate_tensor(s) for s in states]).to(device)
        next_states_tensor = torch.stack([self.inputstate_tensor(s) for s in next_states]).to(device)
        actions_tensor = torch.LongTensor(actions).unsqueeze(1).to(device)
        rewards_tensor = torch.FloatTensor(rewards).to(device)
        dones_tensor = torch.FloatTensor(dones).to(device)

        q_values = self.policy_net(states_tensor).gather(1, actions_tensor).squeeze(1)
        with torch.no_grad():
            next_q_values = target_net(next_states_tensor).max(1)[0]
            expected_q_values = rewards_tensor + gamma * next_q_values * (1 - dones_tensor)

        loss = loss_fn(q_values, expected_q_values)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    def train_dqn(self, episodes, is_slippery=False, render=False):
        self.env = gym.make('FrozenLake-v1', is_slippery=is_slippery, render_mode='human' if render else None)
        self.epsilon = self.epsilon_start

        self.policy_net = DQN(self.state_dim, self.action_dim).to(device)
        target_net = DQN(self.state_dim, self.action_dim).to(device)
        target_net.load_state_dict(self.policy_net.state_dict())

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=0.001)
        self.loss_fn = nn.MSELoss()
        self.replay_buffer = ReplayBuffer(max_size=10000)

        reward_per_episode = np.zeros(episodes)
        step_counter = 0

        for episode in range(episodes):
            state = self.env.reset()[0]
            terminated, truncated = False, False

            while not (terminated or truncated):
                if random.random() < self.epsilon:
                    action = self.env.action_space.sample()
                else:
                    with torch.no_grad():
                        state_tensor = self.inputstate_tensor(state)
                        q_values = self.policy_net(state_tensor)
                        action = q_values.argmax().item()

                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                self.replay_buffer.add_batch(
                    1,
                    [(state, action, reward, next_state, done)]
                )

                state = next_state
                reward_per_episode[episode] += reward
                step_counter += 1

                self.optimize_model(
                    target_net,
                    self.replay_buffer,
                    self.optimizer,
                    self.loss_fn,
                    self.batch_size,
                    self.gamma
                )

                self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

                if step_counter % self.step_count_rate == 0:
                    target_net.load_state_dict(self.policy_net.state_dict())

        self.env.close()
        torch.save(self.policy_net.state_dict(), 'dqn_frozenlake.pth')

    def test_dqn(self, is_slippery=False, episodes=100):
        self.policy_net.eval()
        self.print_dqn(self.policy_net)

        env = gym.make('FrozenLake-v1', is_slippery=is_slippery, render_mode='human')
        total_rewards = []

        for episode in range(episodes):
            state = env.reset()[0]
            terminated, truncated = False, False
            episode_reward = 0

            while not (terminated or truncated):
                with torch.no_grad():
                    state_tensor = self.inputstate_tensor(state).unsqueeze(0)
                    q_values = self.policy_net(state_tensor)
                    action = q_values.argmax().item()

                next_state, reward, terminated, truncated, _ = env.step(action)
                state = next_state
                episode_reward += reward

            total_rewards.append(episode_reward)

        return total_rewards

    def print_dqn(self, dqn):
        num_states = dqn.fc1.in_features

        for s in range(num_states):
            q_vals = dqn(self.inputstate_tensor(s)).tolist()
            q_values = ' '.join(["{:+.2f}".format(q) for q in q_vals])
            best_action = self.ACTIONS[dqn(self.inputstate_tensor(s)).argmax()]
            print(f'{s:02},{best_action},[{q_values}]', end=' ')
            if (s + 1) % 4 == 0:
                print()

In [27]:
main = FrozenLakeEnv()
main.train_dqn(episodes=1000, is_slippery=False, render=False)
main.test_dqn(is_slippery=False, episodes=10)



00,R,[+0.95 +0.94 +0.96 +0.94] 01,R,[+0.95 +0.01 +0.96 +0.96] 02,D,[+0.96 +0.97 +0.96 +0.96] 03,L,[+0.96 -0.09 +0.96 +0.96] 
04,U,[+0.94 +0.94 -0.00 +0.94] 05,U,[+0.68 +0.47 +0.68 +0.73] 06,D,[+0.02 +0.98 -0.00 +0.96] 07,U,[+0.51 +0.62 +0.07 +0.82] 
08,U,[+0.93 +0.13 +0.42 +0.95] 09,L,[+0.95 -0.03 +0.38 +0.09] 10,D,[+0.93 +0.99 -0.02 +0.97] 11,U,[+0.55 +0.25 +0.39 +0.65] 
12,L,[+0.94 +0.78 +0.52 +0.91] 13,U,[+0.80 +0.66 +0.53 +0.95] 14,R,[+0.91 +0.99 +1.00 +0.97] 15,U,[+0.74 +0.56 +0.76 +0.95] 


/home/nikhil/myenv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]